In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
Z = 26
A = 56
mass_nucleon = 0.938273
alpha_fine = 1 / 137.036
nu_axis = True

In [ ]:
def pauli_blocking(ex):
    if ex <= 20:
        return np.exp(-73.3 / ex)
    elif 20 < ex <= 140: 
        return 8.3714e-2 - 9.8343e-3 * ex + 4.1222e-4 * ex**2 - 3.4762e-6 * ex**3 + 9.3537e-9 * ex**4
    else:
        return np.exp(-24.2/ex)
        
def quasi_deuteron(ex):
    N = A - Z
    ex = ex * 1e3
    if ex < 2.224:
        return 0
    else:    
        sigma = 397.8 * (N * Z / A) * ((ex - 2.224)**(3 / 2)) * (ex**-3) * pauli_blocking(ex)
        return sigma * 0.1 * 0.1975**-2

def dipole_E(Q2):
    return 1 / ((1 + Q2 / 0.5)**5)

def QD_suppression(nu, center = 0.12, width = 0.005):
    return 1 / (np.exp((nu - center) / width) + 1)

def RT_quasi_deuteron(nus = [], q2s = [], exs = []):
    nus = np.atleast_1d(nus).astype(float)
    q2s = np.atleast_1d(q2s).astype(float)
    exs = np.atleast_1d(exs).astype(float)

    QDs = [quasi_deuteron(ex) for ex in exs]
    QDs = np.array(QDs)
    
    GEs = dipole_E(q2s)
    GEs = np.array(GEs)
    RTQD = GEs**2 * QDs * nus / (2 * (np.pi**2) * alpha_fine)
    RTQD = RTQD * QD_suppression(nu = nus)
    return RTQD * 1e-3

In [ ]:
# === Input files ===
data_file = "Data/Fe56_RLRT.csv"
fit_dir_qv = "Qvedges"
fit_dir_q2 = "Q2edges"

# Load data
data = pd.read_csv(data_file)

# === Helper to get closest nu from fit ===
def get_fit_values(fit_df, nu):
    idx = (fit_df["nu"] - nu).abs().idxmin()
    return fit_df.loc[idx]

# === Start plotting ===
pdf_name = "Output/Fe56_RLRT_Ratios.pdf"
pp = PdfPages(pdf_name)

rows, cols = 5, 2
plots_per_page = rows * cols

def plot_bin(subset, fit_df, label_var, label_val, fig, axes, plot_count):
    xs, rl_ratios, rl_errs, rt_ratios, rt_errs = [], [], [], [], []

    for _, row in subset.iterrows():
        nu = row["nu"]
        w2 = row["W2"]
        fit_row = get_fit_values(fit_df, nu)

        # Ratios
        rltot = (fit_row["RLQE"] + fit_row["RLIE"] + fit_row["RLE"] + fit_row["RLNS"]) * 1e3 * Z / 6
        rttot = (
            fit_row["RTQE"]
            + fit_row["RTIE"]
            + fit_row["RTE"]
            + fit_row["RTNS"]
            + RT_quasi_deuteron(
                nus=[fit_row["nu"]],
                q2s=[fit_row["q2"]],
                exs=[fit_row["ex"]],
            )[0]
        ) * 1e3 * Z / 6

        rl_ratio = row["RL"] * 1e3 / rltot if rltot != 0 else np.nan
        rt_ratio = row["RT"] * 1e3 / rttot if rttot != 0 else np.nan

        # Errors
        rl_err = row["RLerr"] * 1e3 / rltot if rltot != 0 else np.nan
        rt_err = row["RTerr"] * 1e3 / rttot if rttot != 0 else np.nan

        if nu_axis: 
            xs.append(nu)
        else:
            xs.append(w2)
        rl_ratios.append(rl_ratio)
        rl_errs.append(rl_err)
        rt_ratios.append(rt_ratio)
        rt_errs.append(rt_err)

    # Helper to set x-axis limits (enforcing min=0.05)
    def enforce_xmin(ax, xdata, ydata, label_var, label_val):
        if nu_axis:
            xlim = 0.05
        elif label_var == "qv":
            xlim = mass_nucleon**2 + 2 * mass_nucleon * 0.05 - (label_val**2 - 0.05**2)
        else:
            xlim = mass_nucleon**2 + 2 * mass_nucleon * 0.05 - label_val
        xmin = max(xlim, np.min(xdata))
        if label_val == 0.8:
            xmin *= 1.1
        xmax = np.max(xdata)
        ax.set_xlim(xmin, xmax * 1.05)
        mask = np.array(xdata) >= xmin
        if np.any(mask):
            ymin = np.nanmin(np.array(ydata)[mask])
            ymax = np.nanmax(np.array(ydata)[mask])
            ax.set_ylim(ymin - 0.1 * abs(ymin), ymax + 0.1 * abs(ymax))

    # Plot RL
    ax = axes[plot_count]
    ax.errorbar(xs, rl_ratios, yerr=rl_errs, fmt="o", capsize=3)
    ax.axhline(1, color="gray", linestyle="--")
    enforce_xmin(ax, xs, rl_ratios, label_var, label_val)
    if nu_axis:
        ax.set_xlabel("nu")
    else:
        ax.set_xlabel("W2")
    ax.set_ylabel("ratio")
    ax.text(
        0.05, 0.95, f"{label_var}={label_val}, RL",
        transform=ax.transAxes, ha="left", va="top", fontsize=10, weight="bold"
    )

    # Plot RT
    ax = axes[plot_count + 1]
    ax.errorbar(xs, rt_ratios, yerr=rt_errs, fmt="o", capsize=3)
    ax.axhline(1, color="gray", linestyle="--")
    enforce_xmin(ax, xs, rt_ratios, label_var, label_val)
    if nu_axis:
        ax.set_xlabel("nu")
    else:
        ax.set_xlabel("W2")
    ax.set_ylabel("ratio")
    ax.text(
        0.05, 0.95, f"{label_var}={label_val}, RT",
        transform=ax.transAxes, ha="left", va="top", fontsize=10, weight="bold"
    )

    return plot_count + 2

# Initialize
plot_count = 0
fig = None

# === First handle qvcenter bins ===
for qv in sorted(data.loc[data["qvcenter"].notna(), "qvcenter"].unique()):
    subset = data[(data["qvcenter"] == qv) & data["Q2center"].isna()]
    fit_df = pd.read_csv('Qvedges/' + f'Qvedge_{qv}.csv', index_col = False)

    if plot_count % plots_per_page == 0:
        if fig:
            pp.savefig(fig)
            plt.close(fig)
        fig, axes = plt.subplots(rows, cols, figsize=(12, 18))
        axes = axes.flatten()
        plot_count = 0

    plot_count = plot_bin(subset, fit_df, "qv", qv, fig, axes, plot_count)

# === Then handle Q2center bins ===
for q2 in sorted(data.loc[data["Q2center"].notna(), "Q2center"].unique()):
    subset = data[(data["Q2center"] == q2) & data["qvcenter"].isna()]
    fit_df = pd.read_csv('Q2edges/' + f'Q2edge_{q2}.csv', index_col = False)

    if plot_count % plots_per_page == 0:
        if fig:
            pp.savefig(fig)
            plt.close(fig)
        fig, axes = plt.subplots(rows, cols, figsize=(12, 18))
        axes = axes.flatten()
        plot_count = 0

    plot_count = plot_bin(subset, fit_df, "Q2", q2, fig, axes, plot_count)

# Save last page
if fig:
    pp.savefig(fig)
    plt.close(fig)

pp.close()
print(f"Saved plots to {pdf_name}")